# Ürün Video — Tek Dokunuş Colab\nT4 GPU seçin, aşağıdaki ▶️ düğmesine basın ve istendiğinde ngrok tokenınızı girin.\n

In [ ]:
import os, subprocess, sys, time, threading, getpass
from pathlib import Path
assert subprocess.run(['nvidia-smi'],stdout=subprocess.DEVNULL).returncode==0, 'T4 GPU seçin.'
ROOT=Path('/content/A-V-DEO')
if ROOT.exists(): subprocess.run(['rm','-rf',str(ROOT)],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/efecanca/A-V-DEO.git',str(ROOT)],check=True)
os.chdir(ROOT/'backend'); sys.path.insert(0,str(ROOT/'backend'))
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn[standard]','python-multipart','diffusers>=0.35.1','transformers>=4.51.0','accelerate>=0.34.0','safetensors>=0.4.5','sentencepiece','ftfy','imageio','imageio-ffmpeg','pyngrok','nest_asyncio','huggingface_hub'],check=True)
os.environ['WAN_OFFLOAD_MODE']='sequential'
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Wan-AI/Wan2.1-I2V-14B-480P')
token=getpass.getpass('ngrok authtoken: ').strip()
if not token: raise RuntimeError('ngrok token gerekli')
from pyngrok import ngrok
ngrok.set_auth_token(token)
import nest_asyncio, uvicorn, requests
nest_asyncio.apply()
from main import app
threading.Thread(target=lambda: uvicorn.run(app,host='0.0.0.0',port=8000,log_level='warning'),daemon=True).start()
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health',timeout=3).ok: break
    except: pass
    time.sleep(2)
else: raise RuntimeError('Backend başlatılamadı')
url=ngrok.connect(8000,'http').public_url
print('\n'+'='*60+'\nAPK YE YAZILACAK SUNUCU ADRESİ:\n'+url+'\n'+'='*60)
print('Capabilities HTTP:',requests.get('http://127.0.0.1:8000/capabilities').status_code)


Colab açık kaldığı sürece backend çalışır. İlk model indirmesi ve T4 üzerinde üretim uzun sürebilir.\n